In [0]:
import yaml
from pyspark.sql import functions as F

# Load config
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
silver_schema = "refined"
gold_schema = "gold"

In [0]:
df_market = spark.table(f"{catalog}.{silver_schema}.silver_market_prices")
df_weather = spark.table(f"{catalog}.{silver_schema}.silver_weather")
df_grid = spark.table(f"{catalog}.{silver_schema}.silver_grid_events")
df_ops = spark.table(f"{catalog}.{silver_schema}.silver_regional_operations_base")

In [0]:
# 1. Market Price Pressure
price_threshold = 50  # example threshold
df_market_logic = df_market.withColumn(
    "market_price_pressure",
    F.when(F.col("price") > price_threshold, "HIGH").otherwise("LOW")
)

# 2. Weather Risk Level
df_weather_logic = df_weather.withColumn(
    "weather_risk_level",
    F.when(F.col("wind_speed") > 20, "HIGH")
     .when(F.col("precipitation") > 10, "MEDIUM")
     .otherwise("LOW")
)

# 3. Elevated Grid Incident Indicator
df_grid_logic = df_grid.groupBy("event_day", "region").agg(
    F.max(F.when(F.col("severity_band") == "SEVERE", 1).otherwise(0)).alias("elevated_incident")
)

# 4. Regional Operational Condition
df_ops_logic = df_ops.withColumn(
    "regional_operation_condition",
    F.when(F.col("maintenance_needed") > 5, "ALERT")
     .otherwise("NORMAL")
)

In [0]:
# Combine indicators into a Gold-ready dataframe
df_gold_ready = df_market_logic.join(
    df_weather_logic, on=["region", "report_day"], how="left"
).join(
    df_grid_logic, on=["region", "event_day"], how="left"
).join(
    df_ops_logic, on="region", how="left"
)